# SO-101 Perception Benchmark 2 — methods to beat the ~1cm single-shot ceiling

Benchmark 1 settled the depth model (Depth Pro, ~11 mm floor-anchored, capped by not resolving the 3 cm cube). Here we test two ways past that, and report **x, y, z error separately** so we see which coordinate dominates:
1. **crop-and-zoom** — Depth Pro on an upscaled crop (cube ~26 px → ~400 px).
2. **floor-contact** — SAM 2 (temporary, point-prompted) mask centroid deprojected onto the known floor (z=0 and z=half-height).

Detection is analytic (perfect) for the crop box; floor-contact uses real SAM 2 masks. Run §0, install §1 and **Restart**, then run the rest. Last cell pushes the log + a crop-zoom image.

## 0. Setup + logger

In [ ]:
import os, sys
CLONE_URL = 'https://github.com/Yunsmn/RoboticsPerceptionTest.git'
if not os.path.exists('camera_math.py'):
    os.system('git clone ' + CLONE_URL)
    os.chdir('RoboticsPerceptionTest')
sys.path.insert(0, '.')
import numpy as np, json
from pathlib import Path
from PIL import Image
import camera_math as CM
print('setup OK')

In [ ]:
# Tees every subsequent cell (source + stdout/stderr + tracebacks) to run_log.md.
import sys, io, datetime, traceback, subprocess
from IPython import get_ipython
_LOG = 'run_log.md'; _f = open(_LOG, 'w')
def _w(s=''):
    _f.write(str(s) + '\n'); _f.flush()
_w('# Run log (benchmark2) — ' + datetime.datetime.now().isoformat(timespec='seconds'))
try:
    _gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
except Exception:
    _gpu = ''
_w('- GPU: ' + (_gpu or 'none / CPU')); _w('- Python: ' + sys.version.split()[0])
class _Tee:
    _is_tee = True
    def __init__(self, real): self._real = real
    def write(self, s):
        n = self._real.write(s)
        try: _f.write(s); _f.flush()
        except Exception: pass
        return n
    def flush(self):
        try: self._real.flush()
        except Exception: pass
    def __getattr__(self, k):
        return getattr(self.__dict__['_real'], k)
if not getattr(sys.stdout, '_is_tee', False): sys.stdout = _Tee(sys.stdout)
if not getattr(sys.stderr, '_is_tee', False): sys.stderr = _Tee(sys.stderr)
_ip = get_ipython(); _n = {'i': 0}
def _pre(info):
    _n['i'] += 1
    _w(); _w('## Cell ' + str(_n['i'])); _w('```python')
    _w((info.raw_cell or '').rstrip()); _w('```'); _w('**output:**'); _w('```text')
def _post(res):
    _w('```')
    err = getattr(res,'error_in_exec',None) or getattr(res,'error_before_exec',None)
    if err is not None:
        _w('**ERROR:**'); _w('```text')
        _w(''.join(traceback.format_exception(type(err), err, err.__traceback__))); _w('```')
if not globals().get('_LOGGER_ON'):
    _ip.events.register('pre_run_cell', _pre); _ip.events.register('post_run_cell', _post)
    _LOGGER_ON = True
def download_log():
    _f.flush()
    try:
        from google.colab import files; files.download(_LOG)
    except Exception as e:
        print('download failed (%s) - file is at %s' % (e, _LOG))
print('run logger active -> run_log.md')

## 1. Install (Depth Pro + SAM 2) — then Runtime → Restart

In [ ]:
# Depth Pro (crop-zoom) + SAM 2 (floor-contact masks, via ultralytics). RESTART after install.
# TEMPORARY: using SAM 2 (non-gated) until facebook/sam3 access lands; then swap to SAM 3 (commit d448e15).
get_ipython().system('pip -q install git+https://github.com/apple/ml-depth-pro.git')
get_ipython().system('pip -q install ultralytics')

## 2. Data, floor anchor, cube box

In [ ]:
man = json.loads(Path('data/manifest.json').read_text()); cam = man['camera']
f, cx, cy = cam['f'], cam['cx'], cam['cy']
cam_pos = np.array(cam['cam_pos'], float); R_cw = np.array(cam['R_cw'], float)
W, H = cam['width'], cam['height']; FRAMES = man['frames']
HALF = 0.015                      # cube half-size (~3cm cube); cube centre sits at z=HALF on the floor
CROP_TARGET = 1024                # crop is upscaled so its long side = this many px (cube ~26px -> ~400px)
SAM_MODEL = 'sam2_b.pt'           # TEMPORARY smaller non-gated SAM (auto-downloads); swap to SAM 3 later
print(len(FRAMES), 'frames | %dx%d | f=%.1f' % (W, H, f))

def project(P):
    xc, yc, zc = R_cw.T @ (np.asarray(P, float) - cam_pos)
    return cx + f * xc / -zc, cy - f * yc / -zc, -zc

# harness self-check (same as bench1): project<->deproject must be exact
inv = max(np.linalg.norm(CM.point_at_depth(*project(np.array(fr['gt_xyz']))[:2], f, cx, cy, cam_pos,
          R_cw, project(np.array(fr['gt_xyz']))[2]) - np.array(fr['gt_xyz'])) for fr in FRAMES)
print('self-check project<->deproject max err = %.2e m' % inv)

# known floor pixels (z=0) with analytic true depth -> scale anchor
FLOOR_PTS = []
for vv in range(300, 470, 10):
    for uu in range(40, 600, 20):
        hit = CM.ray_plane(uu, vv, f, cx, cy, cam_pos, R_cw, 0.0)
        if hit is None: continue
        dt = project([hit[0], hit[1], 0.0])[2]
        if 0.5 < dt < 3.0: FLOOR_PTS.append((int(uu), int(vv), float(dt)))
FU = np.array([p[0] for p in FLOOR_PTS]); FV = np.array([p[1] for p in FLOOR_PTS]); FT = np.array([p[2] for p in FLOOR_PTS])
print('floor anchor pixels:', len(FLOOR_PTS))

def cube_box(g, pad=2.5):
    # project the 8 cube corners -> padded square pixel box (analytic 'perfect detection' for the crop)
    cc = [np.array(g) + np.array([sx*HALF, sy*HALF, sz*HALF])
          for sx in (-1,1) for sy in (-1,1) for sz in (-1,1)]
    uv = np.array([project(c)[:2] for c in cc])
    u0, v0 = uv.min(0); u1, v1 = uv.max(0)
    cu, cv = (u0+u1)/2, (v0+v1)/2; s = max(u1-u0, v1-v0) * pad
    return (max(0, int(cu-s/2)), max(0, int(cv-s/2)), min(W, int(cu+s/2)), min(H, int(cv+s/2)))

def box_factor(x, y, half=0.015):
    # silhouette_Δv / (f*h/D) for a box resting on the floor at (x,y) — the elevation geometry.
    # SIZE-INDEPENDENT (verified 1.71 across 3x sizes) so it infers an UNKNOWN height with no size prior.
    g = np.array([x, y, half])
    cc = [g + np.array([sx*half, sy*half, sz*half]) for sx in (-1,1) for sy in (-1,1) for sz in (-1,1)]
    uv = np.array([project(c)[:2] for c in cc]); dv = uv[:,1].max() - uv[:,1].min()
    return dv / (f * (2*half) / project([x, y, 0.0])[2])

## 3. Adapters (Depth Pro with focal override + SAM 2 centroid)

In [ ]:
import torch
_DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', _DEV); _M = {}

def _dp():
    import depth_pro, os
    if 'dp' not in _M:
        ckpt = 'checkpoints/depth_pro.pt'
        if not (os.path.exists(ckpt) and os.path.getsize(ckpt) > 10_000_000):
            os.makedirs('checkpoints', exist_ok=True)
            print('downloading Depth Pro checkpoint (~1.9GB, one-time)...')
            os.system('wget -q https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -O ' + ckpt)
        m, tf = depth_pro.create_model_and_transforms(device=torch.device(_DEV)); _M['dp'] = (m.eval(), tf)
    return _M['dp']

def depth_map(img_rgb, f_px):
    m, tf = _dp()
    with torch.no_grad():
        pred = m.infer(tf(img_rgb), f_px=torch.tensor(float(f_px), device=_DEV))
    return pred['depth'].squeeze().float().cpu().numpy()

_CENT = {}     # cached SAM mask features per frame (run once per image)
def sam_feat(img_rgb, fr):
    # TEMPORARY (SAM 2): point-prompted, seeded with the KNOWN cube pixel (swap to SAM 3 text-prompt,
    # commit d448e15, when access lands). Returns mask centroid + vertical extent + bottom-centre column.
    k = fr['png']
    if k not in _CENT:
        from ultralytics import SAM
        if 'sam' not in _M: _M['sam'] = SAM(SAM_MODEL)      # auto-downloads
        u, v = fr['gt_uv']
        r = _M['sam'](img_rgb, points=[[float(u), float(v)]], labels=[1], verbose=False)
        mk = r[0].masks
        if mk is None or len(mk.data) == 0:
            _CENT[k] = None
        else:
            m = mk.data[0].cpu().numpy() > 0.5
            ys, xs = np.where(m)
            if len(xs) == 0:
                _CENT[k] = None
            else:
                vt, vb = int(ys.min()), int(ys.max())
                ubot = float(xs[ys >= vb - 2].mean())       # bottom-centre column (floor contact)
                _CENT[k] = {'cen': (float(xs.mean()), float(ys.mean())), 'vt': vt, 'vb': vb, 'ubot': ubot}
    return _CENT[k]

## 4. Methods

In [ ]:
# Each method: RGB image + frame -> estimated cube centre (x,y,z) in the base frame, or None.

def m_depthpro_full(img, fr):
    dm = depth_map(img, f)
    fs = float(np.median(FT / dm[FV, FU]))                      # floor-anchor scale (full frame)
    u, v = fr['gt_uv']
    d = float(dm[int(round(v)), int(round(u))]) * fs
    return CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d)

def m_depthpro_crop(img, fr, save_viz=False):
    g = np.array(fr['gt_xyz']); u, v = fr['gt_uv']
    u0, v0, u1, v1 = cube_box(g)
    s = CROP_TARGET / max(u1 - u0, v1 - v0)
    crop = np.array(Image.fromarray(img[v0:v1, u0:u1]).resize((int((u1-u0)*s), int((v1-v0)*s)), Image.BICUBIC))
    dm = depth_map(crop, f * s)                                 # Depth Pro on the zoomed crop, focal scaled
    Hc, Wc = dm.shape
    def cc(uu, vv): return int(round((vv - v0) * s)), int(round((uu - u0) * s))   # -> (row, col)
    # AFFINE calibration: cropping+zooming distorts depth non-uniformly (true ~= a*pred + b, NOT a pure
    # scale). Fit a,b from known floor points inside the crop (robust: reject cube/arm outliers, refit).
    P, T = [], []
    for vv in range(v0, v1, max(1, (v1 - v0) // 14)):
        for uu in range(u0, u1, max(1, (u1 - u0) // 14)):
            hit = CM.ray_plane(uu, vv, f, cx, cy, cam_pos, R_cw, 0.0)
            if hit is None: continue
            dt = project([hit[0], hit[1], 0.0])[2]
            if not (0.5 < dt < 3.0): continue
            r, c = cc(uu, vv)
            if 0 <= r < Hc and 0 <= c < Wc: P.append(float(dm[r, c])); T.append(dt)
    P, T = np.array(P), np.array(T)
    if len(P) < 6: return None
    def fit(Pp, Tt): return np.linalg.lstsq(np.vstack([Pp, np.ones_like(Pp)]).T, Tt, rcond=None)[0]
    a, b = fit(P, T)
    keep = np.abs(a * P + b - T) < 2 * np.median(np.abs(a * P + b - T)) + 1e-6
    if keep.sum() >= 6: a, b = fit(P[keep], T[keep])
    r, c = cc(u, v); d = a * float(dm[min(Hc-1, r), min(Wc-1, c)]) + b
    if save_viz: _viz_crop(crop, dm, cc(u, v), fr, int(keep.sum()))
    return CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d)

def m_floor(img, fr, z_plane):        # deproject centroid onto a FIXED plane (z0 / zhalf reference rows)
    fe = sam_feat(img, fr)
    if fe is None: return None
    hit = CM.ray_plane(fe['cen'][0], fe['cen'][1], f, cx, cy, cam_pos, R_cw, z_plane)
    return None if hit is None else np.array([hit[0], hit[1], z_plane])

def m_floor_inferz(img, fr):          # HONEST: infer the height from the silhouette, no z prior
    fe = sam_feat(img, fr)
    if fe is None: return None
    hit0 = CM.ray_plane(fe['ubot'], fe['vb'], f, cx, cy, cam_pos, R_cw, 0.0)   # floor contact -> (x,y) + depth
    if hit0 is None: return None
    x0, y0 = hit0; D0 = project([x0, y0, 0.0])[2]
    h = ((fe['vb'] - fe['vt']) * D0 / f) / box_factor(x0, y0)                  # infer height (size-free factor)
    zc = h / 2.0
    hit = CM.ray_plane(fe['cen'][0], fe['cen'][1], f, cx, cy, cam_pos, R_cw, zc)   # centroid onto inferred plane
    return None if hit is None else np.array([hit[0], hit[1], zc])

## 5. Crop-zoom depth image (saved)

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt, os
def _viz_crop(crop, dm, rc, fr, nfloor):
    os.makedirs('viz', exist_ok=True)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].imshow(crop); ax[0].set_title('zoomed crop  ' + fr['png']); ax[0].axis('off')
    im = ax[1].imshow(dm, cmap='turbo'); ax[1].set_title('Depth Pro depth on crop (m)'); ax[1].axis('off')
    plt.colorbar(im, ax=ax[1], fraction=0.046)
    for a in ax: a.scatter([rc[1]], [rc[0]], marker='+', s=200, c='lime')
    plt.savefig('viz/bench2_cropzoom.png', dpi=90, bbox_inches='tight'); plt.close()
    print('  saved viz/bench2_cropzoom.png (%d floor px in crop)' % nfloor)

## 6. Evaluate (per-axis)

In [ ]:
def perax(P, G): return np.abs(np.asarray(P, float) - np.asarray(G, float)) * 1000   # [|dx|,|dy|,|dz|] mm

def run_method(name, fn):
    E = []
    for fr in FRAMES:
        img = np.array(Image.open('data/' + fr['png']).convert('RGB'))
        try: P = fn(img, fr)
        except Exception as e:
            if len(E) == 0: print('  %s first-frame error: %s' % (name, e))
            P = None
        if P is not None: E.append(perax(P, fr['gt_xyz']))
    if not E:
        print('%-20s -> NO RESULTS' % name); return {'n': 0}
    E = np.array(E); loc = np.linalg.norm(E, axis=1)
    def q(a, p): return round(float(np.percentile(a, p)), 1)
    r = {'n': len(E),
         'x_med': round(float(np.median(E[:, 0])), 1), 'x_p95': q(E[:, 0], 95),
         'y_med': round(float(np.median(E[:, 1])), 1), 'y_p95': q(E[:, 1], 95),
         'z_med': round(float(np.median(E[:, 2])), 1), 'z_p95': q(E[:, 2], 95),
         'loc_med': round(float(np.median(loc)), 1), 'loc_p95': q(loc, 95)}
    print('%-20s n=%2d | x %4.1f/%-4.1f  y %4.1f/%-4.1f  z %4.1f/%-4.1f  | loc %4.1f/%-4.1f  (med/p95 mm)'
          % (name, r['n'], r['x_med'], r['x_p95'], r['y_med'], r['y_p95'], r['z_med'], r['z_p95'], r['loc_med'], r['loc_p95']))
    return r

# save one crop-zoom depth image on the hero frame first (so we can eyeball that zoom resolves the cube)
_hero = min(FRAMES, key=lambda r: abs(r['gt_xyz'][1]))
try: m_depthpro_crop(np.array(Image.open('data/' + _hero['png']).convert('RGB')), _hero, save_viz=True)
except Exception as e: print('viz skipped:', e)

results = {}
results['depthpro_fullframe'] = run_method('depthpro_full', m_depthpro_full)
results['depthpro_cropzoom']  = run_method('depthpro_crop', m_depthpro_crop)
results['floor_INFERZ']       = run_method('floor_inferz', m_floor_inferz)                        # HONEST height
results['floor_contact_z0']   = run_method('floor_z0',      lambda img, fr: m_floor(img, fr, 0.0))   # no height (ref)
results['floor_contact_zhalf']= run_method('floor_zhalf',   lambda img, fr: m_floor(img, fr, HALF))  # height prior (ceiling ref)

# SAM 3 detection quality (its mask centroid vs the true pixel) — this is the error that feeds
# floor-contact, so report it separately to tell detection error from geometry.
dpx = [np.hypot(fe['cen'][0]-fr['gt_uv'][0], fe['cen'][1]-fr['gt_uv'][1])
       for fr in FRAMES for fe in [_CENT.get(fr['png'])] if fe is not None]
if dpx:
    print('SAM (%s) centroid vs true pixel: median %.1f px | p95 %.1f px | detected %d/%d'
          % (SAM_MODEL, np.median(dpx), np.percentile(dpx, 95), len(dpx), len(FRAMES)))
else:
    print('SAM produced no detections (%s).' % SAM_MODEL)

## 7. Compare

In [ ]:
print('\\n=== benchmark 2: per-axis localisation error (mm), base frame ===')
print('%-22s %8s %8s %8s %10s' % ('method', 'x med/p95', 'y med/p95', 'z med/p95', 'loc med/p95'))
for k, r in results.items():
    if r.get('n'):
        print('%-22s %3.0f/%-4.0f %3.0f/%-4.0f %3.0f/%-4.0f %4.0f/%-5.0f'
              % (k, r['x_med'], r['x_p95'], r['y_med'], r['y_p95'], r['z_med'], r['z_p95'], r['loc_med'], r['loc_p95']))
    else:
        print('%-22s (no results)' % k)
print('\\nbench1 baseline (Depth Pro floor-anchored, 3D only): loc ~11.2mm med / 24.7 p95.')
print('bar: ~2mm. Watch WHICH axis dominates -> tells stage-2 (crop/servo) what to fix.')

## 8. Push the run log

In [ ]:
# Push run_log.md -> logs/run_<timestamp>.md on GitHub (+ viz PNGs). Token from Colab Secrets.
import subprocess, datetime, os, shutil
from google.colab import userdata
print('=== results snapshot ===')
print('results:', globals().get('results', '(eval cell not run yet)'))
def _sh(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret and out: out = out.replace(secret, '***')
    if out: print(out)
    return r.returncode
try:
    _TOKEN = userdata.get('GH_TOKEN')
except Exception as _e:
    _TOKEN = None; print('No GH_TOKEN secret:', _e)
if not _TOKEN:
    print('Set GH_TOKEN in the Colab 🔑 Secrets panel, then re-run. (Fallback: download_log())')
else:
    _REPO = 'Yunsmn/RoboticsPerceptionTest'
    try: _f.flush()
    except Exception: pass
    os.makedirs('logs', exist_ok=True)
    _stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    _dest = 'logs/run2_%s.md' % _stamp
    shutil.copy('run_log.md', _dest)
    _url = 'https://%s@github.com/%s.git' % (_TOKEN, _REPO)
    _sh(['git','config','user.email','younesosf@gmail.com'])
    _sh(['git','config','user.name','Yunsmn'])
    _sh(['git','pull','--rebase','--autostash', _url, 'main'], secret=_TOKEN)
    _sh(['git','add', _dest]); _sh(['git','add', 'viz'])
    _sh(['git','commit','-m','log: benchmark2 run %s' % _stamp])
    _rc = _sh(['git','push', _url, 'HEAD:main'], secret=_TOKEN)
    print(('PUSHED ' if _rc == 0 else 'PUSH FAILED (rc=%d) ' % _rc) + _dest)
    print('-> tell the author: pull and read ' + _dest)